## <center>Importing All Libraries</center>

In [ ]:
import os
import json
import torch
import numpy as np
import matplotlib.pyplot as plt

from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd,
    ScaleIntensityRanged, Resized,
    RandFlipd, RandRotate90d, EnsureTyped,
    Activations, AsDiscrete
)

from monai.data import Dataset, DataLoader, decollate_batch
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric

## <center> Setting up Model configuration</center> 

### <left> Defining Configurations & Data Paths</left>

In [ ]:
DATA_PATH = "data/splits"
IMG_SIZE = (128, 128, 64)
BATCH_SIZE = 1
LR = 1e-4
EPOCHS = 50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

train_files = load_json(os.path.join(DATA_PATH, "train.json"))
val_files = load_json(os.path.join(DATA_PATH, "val.json"))
test_files = load_json(os.path.join(DATA_PATH, "test.json"))

### Defining Transformers

In [ ]:
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),

    ScaleIntensityRanged(
        keys=["image"],
        a_min=0,
        a_max=3000,
        b_min=0.0,
        b_max=1.0,
        clip=True
    ),

    Resized(keys=["image", "label"], spatial_size=IMG_SIZE),

    RandFlipd(keys=["image", "label"], prob=0.5),
    RandRotate90d(keys=["image", "label"], prob=0.5),

    EnsureTyped(keys=["image", "label"])
])

In [ ]:
val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),

    ScaleIntensityRanged(
        keys=["image"],
        a_min=0,
        a_max=3000,
        b_min=0.0,
        b_max=1.0,
        clip=True
    ),

    Resized(keys=["image", "label"], spatial_size=IMG_SIZE),

    EnsureTyped(keys=["image", "label"])
])

## <center>Defining MONAI dataset and Loader </center>

In [5]:
train_ds = Dataset(train_files, transform=train_transforms)
val_ds = Dataset(val_files, transform=val_transforms)

In [6]:
# using dataloader

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2)

In [7]:
# verifying batch check

batch = next(iter(train_loader))

print(batch["image"].shape)
print(batch["label"].shape)

torch.Size([4, 1, 256, 256, 64])
torch.Size([4, 1, 256, 256, 64])


## <center> Model (3D U-Net)</center>

In [ ]:
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(DEVICE)

In [ ]:
loss_fn = DiceCELoss(sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

dice_metric = DiceMetric(include_background=False, reduction="mean")

In [ ]:
post_pred = Compose([
    Activations(sigmoid=True),
    AsDiscrete(threshold=0.5)
])

post_label = AsDiscrete(threshold=0.5)

### Defining Training Loop

In [ ]:
best_dice = 0
train_losses = []
val_dices = []

for epoch in range(EPOCHS):

    model.train()
    epoch_loss = 0

    for batch in train_loader:

        images = batch["image"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    print(f"\nEpoch {epoch+1} | Loss: {avg_loss:.4f}")

    # ---------------- VALIDATION ----------------
    model.eval()
    dice_metric.reset()

    with torch.no_grad():

        for batch in val_loader:

            images = batch["image"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            outputs = model(images)

            outputs = [post_pred(i) for i in decollate_batch(outputs)]
            labels = [post_label(i) for i in decollate_batch(labels)]

            dice_metric(y_pred=outputs, y=labels)

    dice_score = dice_metric.aggregate().item()
    val_dices.append(dice_score)

    print(f"Validation Dice: {dice_score:.4f}")

    # Save best model
    if dice_score > best_dice:
        best_dice = dice_score
        torch.save(model.state_dict(), "best_model.pth")
        print("Best model saved")

Epoch 1, Loss: 0.9494518722806659
Validation Dice: 0.05513051152229309
Best model saved.
Epoch 2, Loss: 0.9406610471861703
Validation Dice: 0.056674957275390625
Best model saved.
Epoch 3, Loss: 0.9333924991743905
Validation Dice: 0.05844844877719879
Best model saved.
Epoch 4, Loss: 0.9282650692122323
Validation Dice: 0.05958496406674385
Best model saved.
Epoch 5, Loss: 0.9243939689227513
Validation Dice: 0.06052505970001221
Best model saved.
Epoch 6, Loss: 0.92147182566779
Validation Dice: 0.06096130982041359
Best model saved.
Epoch 7, Loss: 0.9195461869239807
Validation Dice: 0.06097984313964844
Best model saved.
Epoch 8, Loss: 0.9183345862797329
Validation Dice: 0.06098182499408722
Best model saved.
Epoch 9, Loss: 0.9173316785267421
Validation Dice: 0.06096591427922249
Best model saved.
Epoch 10, Loss: 0.9168865425246102
Validation Dice: 0.06095215678215027
Best model saved.


KeyboardInterrupt: 

## <center> Graphs & Inferences </center>

### <left> Training Curves </left>

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_losses)
plt.title("Training Loss")

plt.subplot(1,2,2)
plt.plot(val_dices)
plt.title("Validation Dice")

plt.show()

### Inference Visualization

In [ ]:
model.eval()

batch = next(iter(val_loader))

images = batch["image"].to(DEVICE)
labels = batch["label"].to(DEVICE)

with torch.no_grad():
    outputs = model(images)
    outputs = torch.sigmoid(outputs)
    outputs = (outputs > 0.5).float()

img = images[0,0].cpu().numpy()
gt = labels[0,0].cpu().numpy()
pred = outputs[0,0].cpu().numpy()

slice_idx = img.shape[-1] // 2

plt.figure(figsize=(15,5))

plt.subplot(1,4,1)
plt.imshow(img[:,:,slice_idx], cmap="gray")
plt.title("MRI")

plt.subplot(1,4,2)
plt.imshow(gt[:,:,slice_idx], cmap="gray")
plt.title("Ground Truth")

plt.subplot(1,4,3)
plt.imshow(pred[:,:,slice_idx], cmap="gray")
plt.title("Prediction")

plt.subplot(1,4,4)
plt.imshow(img[:,:,slice_idx], cmap="gray")
plt.imshow(pred[:,:,slice_idx], alpha=0.4, cmap="Reds")
plt.title("Overlay")

plt.show()